In [3]:
from src.run_app import *
from src.prepare_gam import *
from src.utils import *
from gam_rs_utils.binarize_dataset import binarize_dataset
from gam_rs_utils.utils import *
from FasterRisk.src.fasterrisk import fasterrisk
from time import time

for dataset_name, gap_tolerance in (('bank', 0.005), ('compas', 0.004), ('diabetes', 0.006)):
    path = 'datasets/{}.csv'.format(dataset_name)
    dataset = pd.read_csv(path)
    print(f"Dataset: {dataset_name}")
    print(f"shape: {dataset.shape}")

    results = []
    for num_estimators in [100, 200, 400, 600, -1]:
        if num_estimators == -1:
            dataset.iloc[:, -1] = dataset.iloc[:, -1].replace(0, -1)
            X, y = dataset.iloc[:, :-1], dataset.iloc[:, -1]
            X_one_hot, count = one_hot_encoding(X, one_hot=True)
            y = pd.DataFrame(y)  # {0,1}

            header = list(X_one_hot.columns)
            header = pd.Index(["intercept"] + header)
            header = header.astype("object")

            X_one_hot, y = utils.get_X_y(X_one_hot, y)
            y = y[:, -1]
        else:
            df, thresholds, header, threshold_guess_time = binarize_dataset(dataset, num_estimators)
            X, y = df.iloc[:, :-1], df.iloc[:, -1]

            header = list(X.columns)
            header = pd.Index(["intercept"] + header)
            header = header.astype("object")

            X_one_hot, y = utils.get_X_y(X, y)

        start = time()
        rs = fasterrisk.RiskScoreOptimizer(X_one_hot, y, k=10, lb=-100, ub=100, gap_tolerance=gap_tolerance, select_top_m=-1)
        rs.optimize_with_swaps(swaps=2, fanout_decay=1)
        end = time()
        results.append((dataset_name, len(header), rs.sparseDiversePool_betas.shape[0], end - start))
        print(f"{len(header)} features, {rs.sparseDiversePool_betas.shape[0]} solutions, {end - start:.2f} seconds")

Dataset: bank
shape: (4521, 17)


KeyboardInterrupt: 